# License Plate Detection — EDA
**CMPS 261 — Machine Learning Project**

This notebook is self-contained. It prepares the purified deduplicated dataset first, then explores the selected train/val/test images.


In [ ]:
import sys, os
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
print(f'Running on: {"Google Colab" if IN_COLAB else "Local"}')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    import zipfile
    zip_path = '/content/drive/MyDrive/license_plate_data.zip'
    if not os.path.exists('/content/data/archive'):
        if not os.path.exists(zip_path):
            raise RuntimeError('license_plate_data.zip not found in Google Drive root.')
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall('/content/')
        print('Extracted dataset archive.')
    BASE_DIR = '/content'
else:
    BASE_DIR = '..'

def prepare_purified_dataset(base_dir, seed=42):
    """Create a deduplicated YOLO split from data/archive."""
    import hashlib
    import random
    import shutil
    import xml.etree.ElementTree as ET
    from pathlib import Path

    base_dir = Path(base_dir)
    img_dir = base_dir / 'data' / 'archive' / 'images'
    ann_dir = base_dir / 'data' / 'archive' / 'annotations'
    yolo_dir = base_dir / 'data' / 'yolo'

    if not img_dir.exists() or not ann_dir.exists():
        raise RuntimeError(f'Raw dataset not found under {base_dir / "data" / "archive"}')

    def file_md5(path):
        h = hashlib.md5()
        with open(path, 'rb') as f:
            for chunk in iter(lambda: f.read(1 << 20), b''):
                h.update(chunk)
        return h.hexdigest()

    def parse_xml(xml_path):
        root = ET.parse(xml_path).getroot()
        filename = root.find('filename').text
        img_w = int(root.find('size/width').text)
        img_h = int(root.find('size/height').text)
        boxes = []
        for obj in root.findall('object'):
            boxes.append((
                int(obj.find('bndbox/xmin').text),
                int(obj.find('bndbox/ymin').text),
                int(obj.find('bndbox/xmax').text),
                int(obj.find('bndbox/ymax').text),
            ))
        return filename, img_w, img_h, boxes

    def voc_to_yolo(xmin, ymin, xmax, ymax, img_w, img_h):
        cx = (xmin + xmax) / 2 / img_w
        cy = (ymin + ymax) / 2 / img_h
        w = (xmax - xmin) / img_w
        h = (ymax - ymin) / img_h
        return cx, cy, w, h

    xml_files = sorted(ann_dir.glob('*.xml'))
    hash_to_xmls = {}
    for xml_path in xml_files:
        filename, *_ = parse_xml(xml_path)
        img_path = img_dir / filename
        if img_path.exists():
            hash_to_xmls.setdefault(file_md5(img_path), []).append(xml_path)

    unique_xmls = sorted(min(group) for group in hash_to_xmls.values())
    print(f'Dedup: {len(xml_files)} XMLs -> {len(unique_xmls)} unique images ({len(xml_files) - len(unique_xmls)} duplicate copies removed)')

    random.seed(seed)
    random.shuffle(unique_xmls)
    n = len(unique_xmls)
    n_train = int(n * 0.70)
    n_val = int(n * 0.15)
    splits = {
        'train': unique_xmls[:n_train],
        'val': unique_xmls[n_train:n_train + n_val],
        'test': unique_xmls[n_train + n_val:],
    }

    if yolo_dir.exists():
        shutil.rmtree(yolo_dir)
    for split in splits:
        (yolo_dir / 'images' / split).mkdir(parents=True, exist_ok=True)
        (yolo_dir / 'labels' / split).mkdir(parents=True, exist_ok=True)

    for split, files in splits.items():
        for xml_path in files:
            filename, img_w, img_h, boxes = parse_xml(xml_path)
            shutil.copy2(img_dir / filename, yolo_dir / 'images' / split / filename)
            label_path = yolo_dir / 'labels' / split / f'{Path(filename).stem}.txt'
            with open(label_path, 'w') as f:
                for xmin, ymin, xmax, ymax in boxes:
                    cx, cy, w, h = voc_to_yolo(xmin, ymin, xmax, ymax, img_w, img_h)
                    f.write(f'0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n')

    yaml_path = yolo_dir / 'dataset.yaml'
    yaml_root = str(yolo_dir.resolve()) if str(base_dir) != '/content' else '/content/data/yolo'
    with open(yaml_path, 'w') as f:
        f.write(f'path: {yaml_root}\n')
        f.write('train: images/train\n')
        f.write('val:   images/val\n')
        f.write('test:  images/test\n\n')
        f.write('nc: 1\n')
        f.write("names: ['licence']\n")

    seen = {}
    for split in splits:
        for img in (yolo_dir / 'images' / split).iterdir():
            h = file_md5(img)
            if h in seen and seen[h] != split:
                raise RuntimeError(f'Cross-split duplicate after dedup: {img.name} in {split} matches {seen[h]}')
            seen[h] = split

    print('Data prepared:')
    for split, files in splits.items():
        print(f'  {split:<5}: {len(files)} images')
    print(f'  YAML  : {yaml_path}')
    print('  Cross-split duplicates: 0 (verified)')
    return str(yaml_path)

YAML_PATH = prepare_purified_dataset(BASE_DIR)

import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import random

DATA_DIR = os.path.join(BASE_DIR, 'data', 'archive')
IMG_DIR  = os.path.join(DATA_DIR, 'images')
ANN_DIR  = os.path.join(DATA_DIR, 'annotations')
YOLO_DIR = os.path.join(BASE_DIR, 'data', 'yolo')
RESULTS_DIR = os.path.join(BASE_DIR, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

prepared_files = {}
for split in ['train', 'val', 'test']:
    split_dir = os.path.join(YOLO_DIR, 'images', split)
    for fname in os.listdir(split_dir):
        prepared_files[fname] = split

print(f'Raw image files        : {len(os.listdir(IMG_DIR))}')
print(f'Raw annotations        : {len(os.listdir(ANN_DIR))}')
print(f'Prepared unique images : {len(prepared_files)}')
print(pd.Series(prepared_files).value_counts().rename_axis('split').to_string())


## 1. Parse All Annotations

In [ ]:
records = []

for xml_file in sorted(os.listdir(ANN_DIR)):
    if not xml_file.endswith('.xml'):
        continue
    tree = ET.parse(os.path.join(ANN_DIR, xml_file))
    root = tree.getroot()

    filename  = root.find('filename').text
    if filename not in prepared_files:
        continue

    img_w     = int(root.find('size/width').text)
    img_h     = int(root.find('size/height').text)

    for obj in root.findall('object'):
        xmin = int(obj.find('bndbox/xmin').text)
        ymin = int(obj.find('bndbox/ymin').text)
        xmax = int(obj.find('bndbox/xmax').text)
        ymax = int(obj.find('bndbox/ymax').text)

        box_w  = xmax - xmin
        box_h  = ymax - ymin
        area   = box_w * box_h
        rel_w  = box_w / img_w   # relative width  (0-1)
        rel_h  = box_h / img_h   # relative height (0-1)

        records.append({
            'filename': filename,
            'split': prepared_files[filename],
            'img_w': img_w, 'img_h': img_h,
            'xmin': xmin, 'ymin': ymin, 'xmax': xmax, 'ymax': ymax,
            'box_w': box_w, 'box_h': box_h,
            'area': area,
            'rel_w': rel_w, 'rel_h': rel_h,
            'aspect_ratio': box_w / box_h if box_h > 0 else 0
        })

df = pd.DataFrame(records)
print(f'Total bounding boxes: {len(df)}')
df.head()


## 2. Image Size Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['img_w'], bins=20, color='steelblue', edgecolor='white')
axes[0].set_title('Image Width Distribution')
axes[0].set_xlabel('Width (px)')
axes[0].set_ylabel('Count')

axes[1].hist(df['img_h'], bins=20, color='salmon', edgecolor='white')
axes[1].set_title('Image Height Distribution')
axes[1].set_xlabel('Height (px)')

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'image_size_distribution.png'), dpi=150)
plt.show()

print(df[['img_w', 'img_h']].describe())

## 3. Bounding Box Size & Aspect Ratio

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(df['rel_w'], bins=20, color='steelblue', edgecolor='white')
axes[0].set_title('Relative Box Width')
axes[0].set_xlabel('Width / Image Width')

axes[1].hist(df['rel_h'], bins=20, color='salmon', edgecolor='white')
axes[1].set_title('Relative Box Height')
axes[1].set_xlabel('Height / Image Height')

axes[2].hist(df['aspect_ratio'], bins=20, color='mediumseagreen', edgecolor='white')
axes[2].set_title('Box Aspect Ratio (W/H)')
axes[2].set_xlabel('Aspect Ratio')

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'bbox_distributions.png'), dpi=150)
plt.show()

print(df[['rel_w', 'rel_h', 'aspect_ratio']].describe())

## 4. Bounding Box Center Heatmap
Where do license plates tend to appear in the image?

In [ ]:
cx = ((df['xmin'] + df['xmax']) / 2) / df['img_w']
cy = ((df['ymin'] + df['ymax']) / 2) / df['img_h']

plt.figure(figsize=(6, 5))
plt.hist2d(cx, cy, bins=20, cmap='hot')
plt.colorbar(label='Count')
plt.title('License Plate Center Heatmap (normalized)')
plt.xlabel('Relative X')
plt.ylabel('Relative Y')
plt.gca().invert_yaxis()
plt.savefig(os.path.join(RESULTS_DIR, 'bbox_center_heatmap.png'), dpi=150)
plt.show()

## 5. Sample Images with Bounding Boxes

In [ ]:
sample_pool = df['filename'].unique().tolist()
sample_files = random.sample(sample_pool, min(12, len(sample_pool)))

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

for ax, fname in zip(axes, sample_files):
    img_path = os.path.join(IMG_DIR, fname)
    img = Image.open(img_path).convert('RGB')
    ax.imshow(img)

    rows = df[df['filename'] == fname]
    for _, row in rows.iterrows():
        rect = patches.Rectangle(
            (row['xmin'], row['ymin']),
            row['box_w'], row['box_h'],
            linewidth=2, edgecolor='red', facecolor='none'
        )
        ax.add_patch(rect)

    ax.set_title(fname, fontsize=8)
    ax.axis('off')

plt.suptitle('Sample Images with Ground Truth Bounding Boxes', fontsize=13)
plt.tight_layout()
for ax in axes[len(sample_files):]:
    ax.axis('off')

plt.savefig(os.path.join(RESULTS_DIR, 'sample_images.png'), dpi=150)
plt.show()

## 6. Dataset Summary

In [ ]:
print('=' * 45)
print('DATASET SUMMARY')
print('=' * 45)
print(f'Total images          : {df["filename"].nunique()}')
print(f'Total bounding boxes  : {len(df)}')
print(f'Images with 1 plate   : {(df.groupby("filename").size() == 1).sum()}')
print(f'Images with >1 plate  : {(df.groupby("filename").size() > 1).sum()}')
print(f'Avg image size        : {df["img_w"].mean():.0f} x {df["img_h"].mean():.0f} px')
print(f'Avg box size          : {df["box_w"].mean():.0f} x {df["box_h"].mean():.0f} px')
print(f'Avg relative box size : {df["rel_w"].mean():.2%} x {df["rel_h"].mean():.2%}')
print(f'Avg aspect ratio      : {df["aspect_ratio"].mean():.2f}')
print('Split counts          :')
print(df.drop_duplicates('filename')['split'].value_counts().to_string())